# Study analysis: TestCompass and general-purpose AI

This notebook reproduces the numerical summaries from de-identified, participant-level
inputs for **18 participants: TestCompass (TC), 8; general-purpose AI (AI), 10**.
Task 1 is **MarsRover**; Task 2 is **BowlingScoreKeeper**.

Run the cells in order. Python 3.10+ is sufficient; the analysis uses only the standard
library. Jupyter is needed only for the notebook interface. All paths are relative.
The accompanying `run_notebook.py` also executes the code cells without Jupyter.

## What the data represent

- `quality.json`: counts and ratings for 36 submitted suites, including IntelliJ coverage,
  PIT outcomes, reviewed error counts, and provisional redundant-method counts.
- `activities.json`: coded seconds in eight mutually exclusive activities for each
  participant and task, plus separately recorded uncertain seconds.
- `survey.json`: numerical ratings and reported-use indicators for 18 participants.
- `demographics.json`: aggregated background counts only, without individual profiles.
- `ai_purposes.json`: coded AI-use categories for the ten AI participants in each task.

Participant labels are newly assigned, stable pseudonyms. Names, original identifiers,
raw code submissions, recording filenames, institutional identifiers, free-text responses,
and identity mappings are deliberately absent. These are derived analysis inputs: the
notebook verifies calculations, but cannot independently re-code recordings or rerun
coverage and mutation tools without the private source material.

## Analysis decisions

1. Give every eligible participant equal weight. Calculate rates **within each participant
   and task first**, then average those rates. Do not divide pooled counts or mean minutes
   to obtain the reported percentages.
2. Exclude uncertain intervals from activity numerators and denominators. A participant-task
   with no classified time has an undefined proportion and is excluded from both activity
   means. This produces TC/AI sample sizes of 8/9 for Task 1 and 8/10 for Task 2.
3. Use IntelliJ IDEA coverage agent 1.0.777 for primary line and branch coverage. Retain
   JaCoCo 0.8.15 counts as a sensitivity comparison. PIT 1.30.0 provides mutation results.
4. Assign a mutation score of zero to a suite whose baseline tests failed, as specified
   for this analysis. This is an operational scoring rule, **not a measured PIT score**.
   Show complete-case results separately so the effect of this rule is visible.
5. Count confirmed errors in the error numerator. Keep unclear cases in the denominator;
   show a separate sensitivity calculation treating them as errors.
6. Include all eight TC respondents in tool ratings. One questionnaire used the other
   tool's item wording; the researcher clarified that those responses referred to TC.
7. Analyze final tool-use groups. One participant switched from the originally assigned
   TC condition to general AI, so comparisons of these final groups are not a strict
   intention-to-treat randomized estimate.

Subjective coding for the added participant and the reviewed redundancy classification
remain provisional pending independent researcher review. Counts include retained starter
tests; they do not measure newly authored tests alone.


In [1]:
from pathlib import Path
from fractions import Fraction
from itertools import combinations
from collections import Counter
from math import comb, lcm, floor
import json

# Support launching either from the repository root or from analysis/.
DATA = Path('data') if Path('data/quality.json').exists() else Path('analysis/data')
def load(name):
    return json.loads((DATA / f'{name}.json').read_text())

quality = load('quality')
activities = load('activities')
survey = load('survey')
demographics = load('demographics')
ai_purposes = load('ai_purposes')
TASKS = {1: 'MarsRover', 2: 'BowlingScoreKeeper'}
GROUPS = ('TC', 'AI')

def mean(values):
    values = list(values)
    return sum(values, Fraction(0)) / len(values)

def show(headers, rows):
    """Small, dependency-free table formatter; outputs are aggregate summaries."""
    rows = [[str(value) for value in row] for row in rows]
    widths = [max(len(str(h)), *(len(row[i]) for row in rows))
              for i, h in enumerate(headers)]
    def line(row):
        return ' | '.join(str(value).ljust(width) for value, width in zip(row, widths))
    print(line(headers))
    print('-+-'.join('-' * width for width in widths))
    for row in rows:
        print(line(row))

def fmt(value, digits=3):
    return f'{float(value):.{digits}f}'

assert len(quality) == 36 and len(activities) == 36 and len(survey) == 18
assert Counter(r['condition'] for r in survey) == {'TC': 8, 'AI': 10}
assert len({(r['participant'], r['task']) for r in quality}) == 36
assert len({(r['participant'], r['task']) for r in activities}) == 36
roster = {r['participant']: r['condition'] for r in survey}
for rows in (quality, activities):
    assert all(roster[r['participant']] == r['condition'] for r in rows)
    assert Counter((r['condition'], r['task']) for r in rows) == {
        ('TC', 1): 8, ('TC', 2): 8, ('AI', 1): 10, ('AI', 2): 10}
assert sum(r['test_methods'] for r in quality) == 798
assert sum(r['confirmed_errors'] for r in quality) == 22
assert sum(r['unclear_cases'] for r in quality) == 14
assert sum(r['junit_discovered'] for r in quality) == 797
assert sum(r['junit_failures'] for r in quality) == 3
assert all(r['test_methods'] > 0 for r in quality)
assert all(0 <= r['confirmed_errors'] + r['unclear_cases'] <= r['test_methods'] for r in quality)
assert all(r['redundant_methods'] == r['exact_duplicates'] + r['fully_subsumed'] for r in quality)
print('Validated 18 participants, 36 suites, and 798 inventoried test methods.')


Validated 18 participants, 36 suites, and 798 inventoried test methods.


## Statistical functions and interpretation

The primary comparison is a **two-sided exact participant-level permutation test of
mean differences**, separately for each task. Enumerate every possible group assignment,
retaining the observed group sizes. The p-value is the fraction whose absolute mean
difference is at least as large as observed. Rational arithmetic handles ties exactly;
there is no Monte Carlo approximation or additional “+1” adjustment.

**Cliff's delta** is `P(TC > AI) - P(TC < AI)`, with ties contributing zero. A positive
value means higher TC measurements; whether that is desirable depends on the metric.
It describes pairwise ordering rather than the difference in means.

Apply Holm correction within explicitly stated families: **32 activity comparisons**
(8 activities × 2 tasks × minutes/shares), **20 quality comparisons** (10 measures ×
2 tasks), and **6 survey-rating comparisons**. Full requirement coverage counts are
additional descriptive summaries and are not tested in the quality family. If more
hypotheses are added to a family, its adjusted p-values must be recalculated.

Time shares are continuous participant-level measurements. A two-proportion z-test
would treat them as independent binary trials and is inappropriate. Fisher's exact
test is used only for the explicitly labeled binary helpfulness follow-up.

These small-sample, exploratory comparisons rely on exchangeability of final-group
labels under the null. Switching groups and observational coding limit causal
interpretation. Failure to reject a difference does not demonstrate equivalence.


In [2]:
def permutation_p(tc, ai):
    """Exact two-sided mean-difference test, preserving group sizes."""
    tc, ai = list(map(Fraction, tc)), list(map(Fraction, ai))
    values = tc + ai
    n, total_n = len(tc), len(values)
    scale = lcm(*(value.denominator for value in values))
    weights = [int(value * scale) for value in values]
    total = sum(weights)
    # Multiplying the mean difference by n*(N-n) leaves N*sum(TC)-n*sum(all).
    observed = abs(total_n * sum(weights[:n]) - n * total)
    extreme = sum(abs(total_n * sum(selected) - n * total) >= observed
                  for selected in combinations(weights, n))
    return Fraction(extreme, comb(total_n, n))

def cliffs_delta(tc, ai):
    tc, ai = list(tc), list(ai)
    return Fraction(sum((x > y) - (x < y) for x in tc for y in ai), len(tc) * len(ai))

def holm(p_values):
    """Return adjusted p-values in their original order."""
    adjusted = [Fraction(1)] * len(p_values)
    running_max = Fraction(0)
    for rank, index in enumerate(sorted(range(len(p_values)), key=p_values.__getitem__)):
        running_max = max(running_max, min(Fraction(1), (len(p_values) - rank) * p_values[index]))
        adjusted[index] = running_max
    return adjusted

def compare(tc, ai):
    return {'TC': mean(tc), 'AI': mean(ai), 'delta': cliffs_delta(tc, ai),
            'p': permutation_p(tc, ai)}

def correct(results):
    for result, adjusted in zip(results, holm([r['p'] for r in results])):
        result['p_holm'] = adjusted

def fisher_p(success_tc, total_tc, success_ai, total_ai):
    """Two-sided Fisher exact p-value using probability ordering."""
    total = total_tc + total_ai
    successes = success_tc + success_ai
    def probability(k):
        return Fraction(comb(successes, k) * comb(total-successes, total_tc-k), comb(total, total_tc))
    lo, hi = max(0, total_tc-(total-successes)), min(total_tc, successes)
    observed = probability(success_tc)
    return sum((probability(k) for k in range(lo, hi+1) if probability(k) <= observed), Fraction(0))

# Known small examples check ties, symmetry, direction, and Holm ordering.
assert permutation_p([1, 1], [1, 1]) == 1
assert permutation_p([0, 0], [1, 1]) == Fraction(1, 3)
assert cliffs_delta([0, 0], [1, 1]) == -1
assert holm([Fraction(1, 100), Fraction(4, 100), Fraction(3, 100)]) == [Fraction(3, 100), Fraction(6, 100), Fraction(6, 100)]
print('Exact comparison functions validated.')


Exact comparison functions validated.


## RQ1: Activity allocation

The `seconds` fields are tool consultation (`support`), web search, reference materials,
requirements reading, implementation inspection, test work, JUnit execution/inspection,
and setup/other. Tool consultation combines the assigned tool and any observed use of
another conversational assistant. Test work covers writing, revising, and inspecting tests.

For participant *i*, classified time is the sum of these eight fields. An activity share
is its seconds divided by that participant's classified time. Uncertain intervals are
excluded, not redistributed. For example, a participant with a small amount of classified
time can have a high web-search share despite few minutes; equal participant weighting
preserves that observation.

Displayed percentage columns use largest-remainder rounding to one decimal so they
sum to exactly 100.0%. All tests use the original unrounded fractions. Minute means
are rounded independently. Both raw and Holm-adjusted p-values are shown separately;
only adjusted results should be described as surviving this correction.


In [3]:
ACTIVITY_LABELS = {
    'support': 'Tool consultation', 'web': 'Web search', 'reference': 'References',
    'requirements': 'Requirements', 'implementation': 'Source inspection',
    'test_work': 'Test work', 'junit': 'JUnit execution/inspection', 'other': 'Setup/other'}
for row in activities:
    assert set(row['seconds']) == set(ACTIVITY_LABELS)
    assert all(value >= 0 for value in row['seconds'].values())
    row['classified_seconds'] = sum(row['seconds'].values())
eligible = [r for r in activities if r['classified_seconds'] > 0]
excluded = [r for r in activities if r['classified_seconds'] == 0]
assert len(excluded) == 1 and excluded[0]['condition'] == 'AI' and excluded[0]['task'] == 1
activity_results = []
for task in TASKS:
    groups = {g: [r for r in eligible if r['task'] == task and r['condition'] == g] for g in GROUPS}
    for activity in ACTIVITY_LABELS:
        for outcome in ('minutes', 'percentage'):
            values = {g: [Fraction(r['seconds'][activity], 60) if outcome == 'minutes'
                          else Fraction(100*r['seconds'][activity], r['classified_seconds'])
                          for r in groups[g]] for g in GROUPS}
            activity_results.append(dict(task=task, activity=activity, outcome=outcome,
                                         **compare(values['TC'], values['AI'])))
correct(activity_results)
activity_index = {(r['task'], r['activity'], r['outcome']): r for r in activity_results}

def round_percentages(values):
    tenths = [floor(value * 10) for value in values]
    remainder = 1000 - sum(tenths)
    order = sorted(range(len(values)), key=lambda i: values[i]*10-tenths[i], reverse=True)
    for i in order[:remainder]:
        tenths[i] += 1
    assert sum(tenths) == 1000
    return [f'{value/10:.1f}' for value in tenths]

for task, name in TASKS.items():
    groups = {g: [r for r in eligible if r['task'] == task and r['condition'] == g] for g in GROUPS}
    percentages = {g: round_percentages([activity_index[task, a, 'percentage'][g] for a in ACTIVITY_LABELS]) for g in GROUPS}
    print(f'\n{name}; TC n={len(groups["TC"])}, AI n={len(groups["AI"])}')
    rows = []
    for i, (a, label) in enumerate(ACTIVITY_LABELS.items()):
        result = activity_index[task, a, 'minutes']
        rows.append([label, fmt(result['TC'], 1), percentages['TC'][i], fmt(result['AI'], 1), percentages['AI'][i]])
    rows.append(['Total classified', fmt(mean(Fraction(r['classified_seconds'], 60) for r in groups['TC']), 1),
                 '100.0', fmt(mean(Fraction(r['classified_seconds'], 60) for r in groups['AI']), 1), '100.0'])
    show(['Activity', 'TC min', 'TC %', 'AI min', 'AI %'], rows)
print('\nInference: one Holm family of 32 comparisons')
show(['Task', 'Activity', 'Outcome', 'Raw p', 'Holm p'],
     [[r['task'], ACTIVITY_LABELS[r['activity']], r['outcome'], fmt(r['p'], 6), fmt(r['p_holm'], 6)] for r in activity_results])



MarsRover; TC n=8, AI n=9
Activity                   | TC min | TC %  | AI min | AI % 
---------------------------+--------+-------+--------+------
Tool consultation          | 2.3    | 9.2   | 13.5   | 41.2 
Web search                 | 0.4    | 12.9  | 0.1    | 0.3  
References                 | 0.3    | 0.7   | 0.3    | 1.0  
Requirements               | 0.2    | 0.5   | 3.0    | 9.4  
Source inspection          | 1.3    | 3.1   | 1.8    | 4.6  
Test work                  | 22.3   | 61.0  | 11.1   | 31.8 
JUnit execution/inspection | 2.9    | 9.2   | 3.7    | 9.5  
Setup/other                | 1.0    | 3.4   | 0.8    | 2.2  
Total classified           | 30.7   | 100.0 | 34.3   | 100.0

BowlingScoreKeeper; TC n=8, AI n=10
Activity                   | TC min | TC %  | AI min | AI % 
---------------------------+--------+-------+--------+------
Tool consultation          | 2.5    | 8.2   | 12.4   | 52.6 
Web search                 | 0.9    | 1.8   | 0.6    | 1.6  
References           

### Observed AI-use purposes and verification participation

Purpose categories summarize the existing qualitative coding, not new inference from
screen time. “Multiple purposes” includes test creation plus other assistance (such as
explanation, coverage review, setup, debugging, or submission help); it does not by
itself establish that participants understood the task. A planning-only interaction
requested a checklist and an unfinished test-file skeleton rather than completed tests.
“No observed AI use” is not proof that no off-screen use occurred.

JUnit participation means at least one coded execution/result-inspection interval.
Its denominator is all participants in the final group, unlike activity shares, which
require nonzero classified time.


In [4]:
for task, name in TASKS.items():
    counts = Counter(r['category'] for r in ai_purposes if r['task'] == task)
    assert sum(counts.values()) == 10
    print('\n' + name)
    show(['AI purpose category', 'Participants', '% of all AI participants'],
         [[category, count, f'{100*count/10:.1f}'] for category, count in sorted(counts.items())])
verification_rows = []
for task in TASKS:
    for group in GROUPS:
        rows = [r for r in activities if r['task'] == task and r['condition'] == group]
        count = sum(r['seconds']['junit'] > 0 for r in rows)
        verification_rows.append([TASKS[task], group, f'{count}/{len(rows)}', fmt(Fraction(count*100, len(rows)), 1)])
show(['Task', 'Group', 'Any observed JUnit activity', '%'], verification_rows)



MarsRover
AI purpose category     | Participants | % of all AI participants
------------------------+--------------+-------------------------
Checklist/scaffold only | 1            | 10.0                    
Multiple purposes       | 6            | 60.0                    
No coded AI interaction | 3            | 30.0                    

BowlingScoreKeeper
AI purpose category     | Participants | % of all AI participants
------------------------+--------------+-------------------------
Generation-focused only | 1            | 10.0                    
Multiple purposes       | 8            | 80.0                    
No coded AI interaction | 1            | 10.0                    
Task               | Group | Any observed JUnit activity | %   
-------------------+-------+-----------------------------+-----
MarsRover          | TC    | 7/8                         | 87.5
MarsRover          | AI    | 5/10                        | 50.0
BowlingScoreKeeper | TC    | 7/8                     

## RQ2: Test-suite quality and redundant methods

The quality dictionary stores integer numerators and denominators so ratios can be
reconstructed exactly. Requirement totals are 12 and 13. Test-name quality is the sum
of 1–5 method ratings divided by method count. Issues per method and error percentages
are calculated per participant, then averaged. Lower values are preferable for issues,
errors, and redundancy; more test methods alone do not establish better testing.

“Redundant methods” counts additional methods judged to duplicate or be fully subsumed
by other tests within the same submission, not similar suites across participants.
One representative of a duplicate cluster is retained. Exact duplicates and reviewed
fully subsumed cases are provided separately in the data. This classification is
provisional and is not a claim that all tests touching the same code are redundant.

The primary family contains ten measures for each task. The output lists raw and
Holm-adjusted p-values, avoiding ambiguity about stars in a paper table. The additional
full-requirement-coverage row is descriptive, with delta on its binary indicator.


In [5]:
METRICS = {
    'Requirement coverage (%)': lambda r: Fraction(100*r['requirements_met'], r['requirements_total']),
    'Line coverage (%)': lambda r: Fraction(100*r['lines_covered'], r['lines_total']),
    'Branch coverage (%)': lambda r: Fraction(100*r['branches_covered'], r['branches_total']),
    'Mutation score (%)': lambda r: Fraction(100*r['mutants_killed'], r['mutants_generated']) if r['mutation_baseline_passed'] else Fraction(0),
    'Maintainability issues': lambda r: Fraction(r['maintainability_issues']),
    'Issues per test method': lambda r: Fraction(r['maintainability_issues'], r['test_methods']),
    'Test-name quality (1-5)': lambda r: Fraction(r['test_name_score_sum'], r['test_methods']),
    'Inventoried test methods': lambda r: Fraction(r['test_methods']),
    'Erroneous tests (%)': lambda r: Fraction(100*r['confirmed_errors'], r['test_methods']),
    'Redundant test methods': lambda r: Fraction(r['redundant_methods'])}
quality_results = []
for task in TASKS:
    groups = {g: [r for r in quality if r['task'] == task and r['condition'] == g] for g in GROUPS}
    for label, measure in METRICS.items():
        values = {g: [measure(r) for r in groups[g]] for g in GROUPS}
        quality_results.append(dict(task=task, measure=label, **compare(values['TC'], values['AI'])))
correct(quality_results)
for task, name in TASKS.items():
    print('\n' + name)
    show(['Measure', 'TC mean', 'AI mean', 'Delta', 'Raw p', 'Holm p', 'Holm < .05'],
         [[r['measure'], fmt(r['TC'], 3), fmt(r['AI'], 3), fmt(r['delta'], 3), fmt(r['p'], 6),
           fmt(r['p_holm'], 6), '*' if r['p_holm'] < Fraction(5, 100) else '']
          for r in quality_results if r['task'] == task])
    binary = {g: [int(r['requirements_met'] == r['requirements_total']) for r in quality
                  if r['condition'] == g and r['task'] == task] for g in GROUPS}
    print('Full requirement coverage (descriptive): ' + '; '.join(
        f'{g} {sum(values)}/{len(values)} ({100*sum(values)/len(values):.1f}%)' for g, values in binary.items())
        + f'; delta={float(cliffs_delta(binary["TC"], binary["AI"])):.2f}')



MarsRover
Measure                  | TC mean | AI mean | Delta  | Raw p    | Holm p   | Holm < .05
-------------------------+---------+---------+--------+----------+----------+-----------
Requirement coverage (%) | 85.417  | 90.833  | 0.000  | 0.516386 | 1.000000 |           
Line coverage (%)        | 88.051  | 98.235  | -0.212 | 0.066731 | 1.000000 |           
Branch coverage (%)      | 72.266  | 87.656  | -0.650 | 0.022122 | 0.376068 |           
Mutation score (%)       | 79.245  | 84.906  | -0.250 | 0.715161 | 1.000000 |           
Maintainability issues   | 2.875   | 4.800   | -0.062 | 0.754719 | 1.000000 |           
Issues per test method   | 0.359   | 0.195   | 0.425  | 0.455780 | 1.000000 |           
Test-name quality (1-5)  | 3.062   | 3.705   | -0.450 | 0.082979 | 1.000000 |           
Inventoried test methods | 12.000  | 28.300  | -0.800 | 0.007976 | 0.143562 |           
Erroneous tests (%)      | 0.000   | 0.303   | -0.100 | 1.000000 | 1.000000 |           
Redundant 

### Quality sensitivity checks

Three suites failed PIT baseline tests: one AI MarsRover suite and two TC bowling suites.
The complete-case mutation means below omit these failures; the primary scores include
them as zero. An AI bowling submission needed supplemental production classes; its
exclusion is shown separately. These are descriptive sensitivity analyses, not extra
significance searches. Changing scoring rules for inference requires recomputing the
whole prespecified comparison family.

Treating unclear cases as errors gives coding-sensitivity bounds, not confidence
intervals. Passing execution does not guarantee a correct oracle; runtime failures
also do not automatically establish an erroneous test.


In [6]:
sensitivity_rows = []
for task in TASKS:
    for group in GROUPS:
        rows = [r for r in quality if r['task'] == task and r['condition'] == group]
        valid = [r for r in rows if r['mutation_baseline_passed']]
        sensitivity_rows.append([TASKS[task], group, f'{len(valid)}/{len(rows)}',
            fmt(mean(METRICS['Mutation score (%)'](r) for r in rows), 2),
            fmt(mean(METRICS['Mutation score (%)'](r) for r in valid), 2),
            fmt(mean(METRICS['Erroneous tests (%)'](r) for r in rows), 2),
            fmt(mean(Fraction(100*(r['confirmed_errors']+r['unclear_cases']), r['test_methods']) for r in rows), 2)])
show(['Task', 'Group', 'Valid PIT n/all', 'Mutation incl. zero', 'Mutation valid only', 'Confirmed error %', 'Error % incl. unclear'], sensitivity_rows)
line_gaps = [abs(Fraction(100*r['lines_covered'], r['lines_total']) -
                 Fraction(100*r['jacoco_lines_covered'], r['jacoco_lines_total'])) for r in quality]
branch_agreement = sum(Fraction(r['branches_covered'], r['branches_total']) ==
                       Fraction(r['jacoco_branches_covered'], r['jacoco_branches_total']) for r in quality)
print(f'\nIntelliJ vs JaCoCo: branches agree in {branch_agreement}/36 suites; maximum line gap = {float(max(line_gaps)):.3f} percentage points.')
for task in TASKS:
    for group in GROUPS:
        original = [r for r in quality if r['task'] == task and r['condition'] == group]
        restricted = [r for r in original if not r['supplemental_production_classes']]
        if len(restricted) != len(original):
            print(f'\nExcluding supplemental production classes: {TASKS[task]}, {group}, n={len(restricted)}')
            show(['Measure', 'All suites', 'Excluding supplemental'],
                 [[label, fmt(mean(METRICS[label](r) for r in original), 2), fmt(mean(METRICS[label](r) for r in restricted), 2)]
                  for label in ['Line coverage (%)', 'Branch coverage (%)', 'Mutation score (%)']])


Task               | Group | Valid PIT n/all | Mutation incl. zero | Mutation valid only | Confirmed error % | Error % incl. unclear
-------------------+-------+-----------------+---------------------+---------------------+-------------------+----------------------
MarsRover          | TC    | 8/8             | 79.25               | 79.25               | 0.00              | 4.51                 
MarsRover          | AI    | 9/10            | 84.91               | 94.34               | 0.30              | 1.57                 
BowlingScoreKeeper | TC    | 6/8             | 66.59               | 88.78               | 10.96             | 12.69                
BowlingScoreKeeper | AI    | 10/10           | 91.54               | 91.54               | 2.63              | 3.99                 

IntelliJ vs JaCoCo: branches agree in 36/36 suites; maximum line gap = 1.249 percentage points.

Excluding supplemental production classes: BowlingScoreKeeper, AI, n=9
Measure             | All suites 

## RQ3: Survey ratings and reported use

All six ratings use a 1–5 scale. Report **means**, with TC n=8 and AI n=10 throughout.
The six between-group comparisons share a Holm family. Pre/post change counts are
paired within participants and reported descriptively; comparing the two groups'
post ratings is not a test of change within either group.

The helpfulness ≥4 comparison is an exploratory binary follow-up with a two-sided
Fisher exact test, shown as **unadjusted** and distinguished from the six rating tests.
Reported-use selections permit multiple answers; their percentages need not sum to 100%.
The AI questionnaire retained its original ChatGPT wording even when other assistants
were used. One TC respondent's tool-specific answers were reassigned to TC following
researcher clarification, without imputing new ratings.


In [7]:
SURVEY_ITEMS = {'helpfulness': 'Helpfulness', 'recommend': 'Recommend',
    'knowledge_pre': 'Knowledge, pre', 'knowledge_post': 'Knowledge, post',
    'confidence_pre': 'Confidence, pre', 'confidence_post': 'Confidence, post'}
survey_results = []
for item, label in SURVEY_ITEMS.items():
    values = {g: [Fraction(r[item]) for r in survey if r['condition'] == g] for g in GROUPS}
    survey_results.append(dict(measure=label, **compare(values['TC'], values['AI'])))
correct(survey_results)
show(['Rating', 'TC mean', 'AI mean', 'Raw p', 'Holm p'],
     [[r['measure'], fmt(r['TC'], 2), fmt(r['AI'], 2), fmt(r['p'], 6), fmt(r['p_holm'], 6)] for r in survey_results])
print('\nWithin-person changes (descriptive)')
change_rows = []
for group in GROUPS:
    for item in ('knowledge', 'confidence'):
        changes = [r[item+'_post']-r[item+'_pre'] for r in survey if r['condition'] == group]
        change_rows.append([group, item, sum(v > 0 for v in changes), sum(v < 0 for v in changes), sum(v == 0 for v in changes)])
show(['Group', 'Rating', 'Increased', 'Decreased', 'Unchanged'], change_rows)
print('\nReported uses (descriptive counts and percentages)')
use_rows = []
for item in survey[0]['reported_use']:
    values = []
    for group in GROUPS:
        rows = [r for r in survey if r['condition'] == group]
        count = sum(r['reported_use'][item] for r in rows)
        values.append(f'{count}/{len(rows)} ({100*count/len(rows):.1f}%)')
    use_rows.append([item.replace('_', ' '), *values])
show(['Use', 'TC', 'AI'], use_rows)
counts = {g: sum(r['helpfulness'] >= 4 for r in survey if r['condition'] == g) for g in GROUPS}
print(f'\nHelpfulness >=4: TC {counts["TC"]}/8; AI {counts["AI"]}/10; unadjusted Fisher p={float(fisher_p(counts["TC"], 8, counts["AI"], 10)):.6f}')
for group in GROUPS:
    rows = [r for r in survey if r['condition'] == group]
    recommend = sum(r['recommend'] >= 4 for r in rows)
    mismatch = sum(r['helpfulness'] >= 4 and r['recommend'] <= 2 for r in rows)
    print(f'{group}: recommendation >=4 = {recommend}/{len(rows)} ({100*recommend/len(rows):.1f}%); helpfulness >=4 but recommendation <=2 = {mismatch}/{len(rows)}.')


Rating           | TC mean | AI mean | Raw p    | Holm p  
-----------------+---------+---------+----------+---------
Helpfulness      | 3.25    | 4.70    | 0.003337 | 0.020019
Recommend        | 3.38    | 3.60    | 0.871521 | 1.000000
Knowledge, pre   | 3.00    | 3.00    | 1.000000 | 1.000000
Knowledge, post  | 2.50    | 2.70    | 0.704374 | 1.000000
Confidence, pre  | 3.00    | 2.90    | 1.000000 | 1.000000
Confidence, post | 2.88    | 2.90    | 1.000000 | 1.000000

Within-person changes (descriptive)
Group | Rating     | Increased | Decreased | Unchanged
------+------------+-----------+-----------+----------
TC    | knowledge  | 0         | 4         | 4        
TC    | confidence | 2         | 3         | 3        
AI    | knowledge  | 1         | 3         | 6        
AI    | confidence | 1         | 1         | 8        

Reported uses (descriptive counts and percentages)
Use                   | TC          | AI          
----------------------+-------------+-------------
what to

## Demographics (aggregated)

Background categories are counts from the post-study survey. Experience was collected
in ranges, so it cannot be converted to an exact mean number of years. Individual
background profiles and free-text responses are not published. One TC participant's
graduate classification follows the response recorded in the survey and remains a
point for researcher verification.


In [8]:
for measure in dict.fromkeys(r['measure'] for r in demographics):
    rows = [r for r in demographics if r['measure'] == measure]
    assert sum(r['TC'] for r in rows) == 8 and sum(r['AI'] for r in rows) == 10
show(['Measure', 'Category', 'TC', 'AI'],
     [[r['measure'], r['category'], r['TC'], r['AI']] for r in demographics])


Measure                  | Category               | TC | AI
-------------------------+------------------------+----+---
Education                | Graduate               | 1  | 0 
Education                | Undergraduate          | 7  | 10
Gender                   | Man                    | 6  | 5 
Gender                   | Non-Binary             | 0  | 1 
Gender                   | Prefer not to disclose | 1  | 0 
Gender                   | Woman                  | 1  | 4 
Programming experience   | 1 year- 3 years        | 3  | 6 
Programming experience   | Less than 1 year       | 2  | 1 
Programming experience   | More than 3 years      | 3  | 3 
Java experience          | 1 year - 3 years       | 4  | 3 
Java experience          | Less than 1 year       | 4  | 4 
Java experience          | More than 3 years      | 0  | 3 
Unit-testing experience  | Less than 1 year       | 8  | 10
Programming proficiency  | Advanced beginner      | 4  | 3 
Programming proficiency  | Competent    

## Reproducibility checks and reporting

The checks below guard against accidentally returning to pooled percentages, excluding
failed mutation suites from the primary score, or dropping the clarified TC survey
response. Exact inferential results should be computed from the numerical inputs,
not reconstructed from rounded paper tables.

Report a raw p-value as unadjusted. For corrected significance, use the Holm column and
state the comparison family. The current family definitions are analysis choices made
for this report; this notebook does not claim they were preregistered. Qualitative
interpretations and causal or educational claims require evidence beyond these tests.


In [9]:
qindex = {(r['task'], r['measure']): r for r in quality_results}
assert abs(float(qindex[1, 'Mutation score (%)']['AI']) - 84.9056603774) < 1e-6
assert abs(float(qindex[2, 'Mutation score (%)']['TC']) - 66.5865384615) < 1e-6
assert qindex[1, 'Redundant test methods']['delta'] == Fraction(-1, 5)
assert qindex[2, 'Redundant test methods']['delta'] == Fraction(-9, 80)
assert len(activity_results) == 32 and len(quality_results) == 20 and len(survey_results) == 6
assert not any(r['p_holm'] < Fraction(5, 100) for r in activity_results)
assert {(r['task'], r['measure']) for r in quality_results if r['p_holm'] < Fraction(5, 100)} == {
    (2, 'Inventoried test methods'), (2, 'Issues per test method')}
assert [r['measure'] for r in survey_results if r['p_holm'] < Fraction(5, 100)] == ['Helpfulness']
assert survey_results[0]['TC'] == Fraction(13, 4)
assert float(activity_index[1, 'web', 'percentage']['TC']) > 12
print('All summary checks passed. The notebook uses participant-level means throughout.')


All summary checks passed. The notebook uses participant-level means throughout.
